# Student Performance Prediction

**Machine Learning Project — Mathematics Dataset**

**Team:** Tanmay Raghav, Aditya Sirohi, Ritivik Sharma  
**Submitted to:** Professor Nirbhay  
**Academic year:** 2026

This notebook uses student information to estimate the final Mathematics grade (G3). It covers data checking, graphs, preprocessing, model training and evaluation.

We use two models: Linear Regression and Random Forest.

## 1. Import libraries

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid")

## 2. Load the dataset

The notebook first checks for a local CSV file. If it is not present (for example, in a fresh Colab session), it downloads the Mathematics file from this project's GitHub repository.

In [ ]:
# Find the dataset in either the repository root or the notebooks folder.
DATA_CANDIDATES = [
    Path("data/student-mat.csv"),
    Path("../data/student-mat.csv"),
]
DATA_PATH = next((path for path in DATA_CANDIDATES if path.exists()), None)

# Colab can download the CSV directly from GitHub when no local copy is present.
DATA_URL = "https://raw.githubusercontent.com/Tanmayraghav/DATA-SCIENCE-/main/data/student-mat.csv"

if DATA_PATH is not None:
    df = pd.read_csv(DATA_PATH, sep=";")
else:
    df = pd.read_csv(DATA_URL, sep=";")

print("Dataset loaded successfully")
print("Rows and columns:", df.shape)
display(df.head())

## 3. Check the data

Before making graphs or training a model, we check the column types, missing values and duplicate records.

In [ ]:
print("Data types:")
print(df.dtypes)

print("\nMissing values in each column:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())
df = df.drop_duplicates().copy()
print("Shape after checking duplicates:", df.shape)

## 4. Exploratory data analysis

These graphs give a first look at the grades and some of the other columns. They show patterns in this dataset; they do not prove that one factor causes a change in marks.

### 4.1 Distribution of final grades (G3)

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(df["G3"], bins=15, edgecolor="black")
plt.title("Distribution of Final Mathematics Grades")
plt.xlabel("Final grade (G3)")
plt.ylabel("Number of students")
plt.tight_layout()
plt.show()

### 4.2 Study time and final grade

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(data=df, x="studytime", y="G3", errorbar=None)
plt.title("Average Final Grade by Study-Time Category")
plt.xlabel("Weekly study-time category")
plt.ylabel("Average final grade (G3)")
plt.tight_layout()
plt.show()

print("Average G3 by study-time category:")
print(df.groupby("studytime")["G3"].mean())

### 4.3 Second-period grade (G2) and final grade (G3)

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df["G2"], df["G3"], alpha=0.7)
plt.title("Second-Period Grade and Final Grade")
plt.xlabel("Second-period grade (G2)")
plt.ylabel("Final grade (G3)")
plt.tight_layout()
plt.show()

### 4.4 Absences

In [ ]:
plt.figure(figsize=(8, 3))
sns.boxplot(x=df["absences"])
plt.title("Absences: Box Plot")
plt.xlabel("Number of absences")
plt.tight_layout()
plt.show()

# We inspect possible outliers but do not remove records automatically.

### 4.5 Correlation between numerical columns

In [ ]:
numeric_df = df.select_dtypes(include=np.number)
plt.figure(figsize=(12, 8))
sns.heatmap(numeric_df.corr(), cmap="coolwarm", center=0)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

## 5. Prepare the data

The target is G3. We keep G1 and G2 as input columns because this project assumes the earlier-period grades are known when making the estimate. The data is split before encoding and scaling.

In [ ]:
X = df.drop(columns=["G3"])
y = df["G3"]

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training records:", len(X_train_raw))
print("Testing records:", len(X_test_raw))

### 5.1 Convert categorical columns into numbers

In [ ]:
X_train = pd.get_dummies(X_train_raw, drop_first=True)
X_test = pd.get_dummies(X_test_raw, drop_first=True)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

print("Training feature shape:", X_train.shape)
print("Testing feature shape:", X_test.shape)

### 5.2 Scale the features for Linear Regression

The scaler is fitted only on the training data. Random Forest will use the encoded data without scaling.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("Feature scaling completed.")

## 6. Train the models

Both models use the same training and test records. Random Forest uses 200 trees.

In [ ]:
linear_model = LinearRegression()
linear_model.fit(X_train_scaled, y_train)
linear_predictions = linear_model.predict(X_test_scaled)

random_forest_model = RandomForestRegressor(n_estimators=200, random_state=42)
random_forest_model.fit(X_train, y_train)
random_forest_predictions = random_forest_model.predict(X_test)

print("Both models have been trained.")

## 7. Evaluate and compare the models

MAE, MSE and RMSE measure prediction error, so lower values are better. R² is generally better when it is higher.

In [ ]:
def calculate_metrics(model_name, actual, predicted):
    mse = mean_squared_error(actual, predicted)
    return {
        "Model": model_name,
        "MAE": mean_absolute_error(actual, predicted),
        "MSE": mse,
        "RMSE": np.sqrt(mse),
        "R2": r2_score(actual, predicted),
    }

comparison = pd.DataFrame([
    calculate_metrics("Linear Regression", y_test, linear_predictions),
    calculate_metrics("Random Forest", y_test, random_forest_predictions),
]).sort_values("RMSE").reset_index(drop=True)

display(comparison.round(3))

## 8. Compare model errors

In [ ]:
error_data = comparison.melt(
    id_vars="Model", value_vars=["MAE", "RMSE"],
    var_name="Metric", value_name="Error"
)

plt.figure(figsize=(8, 5))
sns.barplot(data=error_data, x="Model", y="Error", hue="Metric")
plt.title("Model Error Comparison")
plt.ylabel("Error (lower is better)")
plt.tight_layout()
plt.show()

## 9. Actual grades and predicted grades

In [ ]:
predictions = {
    "Linear Regression": linear_predictions,
    "Random Forest": random_forest_predictions,
}

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (model_name, predicted) in zip(axes, predictions.items()):
    ax.scatter(y_test, predicted, alpha=0.7)
    lower = min(float(y_test.min()), float(np.min(predicted)))
    upper = max(float(y_test.max()), float(np.max(predicted)))
    ax.plot([lower, upper], [lower, upper], "k--", linewidth=1)
    ax.set_xlabel("Actual G3")
    ax.set_ylabel("Predicted G3")
    ax.set_title(model_name)

fig.suptitle("Actual vs Predicted Final Grades")
plt.tight_layout()
plt.show()

## 10. Sample predictions

In [ ]:
best_model_name = comparison.loc[0, "Model"]
best_predictions = predictions[best_model_name]

sample_results = pd.DataFrame({
    "Actual": y_test.to_numpy(),
    "Predicted": best_predictions,
}, index=y_test.index)

print("Sample predictions from", best_model_name)
display(sample_results.head(10).round(2))
print("Lowest test RMSE in this comparison:", best_model_name)

## 11. Conclusion

In this run, Random Forest had a lower test error than Linear Regression. This result applies to this train-test split; it does not mean Random Forest will always be better. The dataset has 395 Mathematics records, and G1 and G2 are included as inputs. A separate experiment would be needed to estimate G3 without earlier grades.

The project is for learning and should not be used alone to make important decisions about individual students.

**Dataset:** [UCI Student Performance](https://archive.ics.uci.edu/dataset/320/student+performance)  
**Citation:** Cortez, P. (2008). *Student Performance* [Dataset]. https://doi.org/10.24432/C5TG7T.